# Quantum Evidence Lab

A free local walkthrough accompanying the research blueprint. All supplied outcomes are simulated; there are no student or QPU results. Start with README.md. Run from the extracted package directory. The core environment needs NumPy and SciPy; no Qiskit account is required.

## 1 A test can pass without supporting the claim
The target is Phi-plus. Phi-minus changes relative phase. Predict whether ordinary Z-basis measurements can distinguish them before running the cell.

In [1]:
import json
from pathlib import Path
import numpy as np
from experiment import cases, properties, H
lookup = {name: rho for name, rho, kind in cases()}
for name in ['target', 'phase_flip', 'dephased']:
    rho = lookup[name]
    fidelity, q = properties(rho)
    print(name, 'Z probabilities:', np.diag(rho).real.round(3).tolist(),
          'X odd-parity probability:', round(q['XX'], 3), 'fidelity:', round(fidelity, 3))


target Z probabilities: [0.5, 0.0, 0.0, 0.5] X odd-parity probability: 0.0 fidelity: 1.0
phase_flip Z probabilities: [0.5, 0.0, 0.0, 0.5] X odd-parity probability: 1.0 fidelity: 0.0
dephased Z probabilities: [0.5, 0.0, 0.0, 0.5] X odd-parity probability: 0.5 fidelity: 0.5


All three have the same Z distribution. The X evidence distinguishes them. More Z-only samples cannot supply the missing information. Testing a specified output does not establish equivalence of programs for every input.

## 2 Evidence can support a bounded claim
The chosen classroom contract is fidelity at least 0.90. The code uses confidence intervals and permits an inconclusive decision. The method is built on established state-verification mathematics, not a new quantum algorithm.

In [2]:
from experiment import cp_table, fidelity_bounds, verdict
n = 192
lo, hi = cp_table(n, 0.05 / 6)
intervals = {'XX': (float(lo[0]), float(hi[0])), 'ZZ': (float(lo[0]), float(hi[0]))}
lower, upper = fidelity_bounds(intervals)
print('After 384 total shots, no observed parity failures:')
print('Fidelity interval:', round(lower, 4), round(upper, 4))
print('Decision:', verdict(lower, upper))


After 384 total shots, no observed parity failures:
Fidelity interval: 0.9437 1.0
Decision: supported


## 3 Read the completed experiment
There are 17 fixtures and 400 repetition seeds per fixture, with shared observations for paired policy comparisons. A policy gets at most 1536 shots. Repeated trials are not independent circuit families or students.

In [3]:
results = json.loads(Path('results/results.json').read_text())
print('Python:', results['python'], 'NumPy:', results['numpy'], 'SciPy:', results['scipy'])
for name, row in results['overall'].items():
    print(name, '| wrong:', row['wrong_decisions'], '/', row['n'],
          '| inconclusive:', f"{100 * row['inconclusive']/row['n']:.2f}%",
          '| mean shots:', round(row['mean_shots'], 1))


Python: 3.12.14 NumPy: 2.3.5 SciPy: 1.17.0
z_only_teaching_negative_control | wrong: 2597 / 6800 | inconclusive: 0.00% | mean shots: 1536.0
fixed_XX_ZZ | wrong: 7 / 6800 | inconclusive: 35.37% | mean shots: 1536.0
staged_XX_ZZ | wrong: 5 / 6800 | inconclusive: 37.81% | mean shots: 800.7
fixed_XX_YY_ZZ | wrong: 0 / 6800 | inconclusive: 32.57% | mean shots: 1536.0


The staged two-basis method saves shots but abstains more often. The three-basis method resolves more cases. This is not evidence that one policy is universally best. A scientifically honest report includes wrong decisions and inconclusive outcomes.

## 4 Check the independent Qiskit calculation
The optional Qiskit script checks all 51 case/basis probabilities. Its counts are from a seeded local simulator, not physical quantum hardware.

In [4]:
check = json.loads(Path('results/qiskit_check.json').read_text())
print('Qiskit version:', check['qiskit'])
print('Probability checks:', check['basis_checks'])
print('Maximum absolute discrepancy:', check['max_absolute_probability_error'])
for name, bases in check['example'].items():
    print(name, {basis: value['counts'] for basis, value in bases.items()})


Qiskit version: 2.5.2
Probability checks: 51
Maximum absolute discrepancy: 3.3306690738754696e-16
target {'Z': {'00': 131, '11': 125}, 'X': {'00': 131, '11': 125}}
phase_flip {'Z': {'00': 131, '11': 125}, 'X': {'01': 131, '10': 125}}


## Reproduce and extend

Run `python experiment.py --out my_results` to reproduce the experiment. Use `--seed 123` to explore another realization and report it as a new run. Run `python qiskit_example.py` in the separate Qiskit environment to repeat that check.

Try changing the measurement budget in a copy of the script. Specify hypotheses before comparing outcomes. Do not add unplanned intermediate looks without correcting the inference.

Use LAB_HANDOUT.md for a lesson and STUDY_PROTOCOL.md for the proposed human study. No educational benefit has yet been measured.